# GRN inference

Infers context-specific transcription factor (TF) → target gene regulatory edges for the two stimulus contexts:

| Context | Cells | Upstream signal seeding the TF ranking |
|---|---|---|
| `K562` | K562 + TNF-α | TNF (ligand) |
| `THP1` | THP1 + LPS | TLR4 (receptor) |

The workflow has three steps:

1. **Rank TFs** (`cp.tl.grn.rank_regulators`): signed diffusion over the OmniPath signaling network, starting from each context's upstream signal. TFs (human TFs that have OmniPath transcriptional targets) are absorbing nodes.
2. **Score target genes** (`cp.tl.grn.score_targets`): recalibrate each context's tradeSeq association p-values into local FDRs with fdrtool (R backend).
3. **Infer edges** (`cp.tl.grn.infer`): combine TF priorities with target-gene scores into ranked TF → target edges.

OmniPath is queried through the `omnipath` client, which caches responses under `~/.cache/omnipathdb`.


In [ ]:
from pathlib import Path

import chronopack as cp


## Configuration

Inputs, contexts and output locations. Paths are relative to this notebook's directory (`grn_inference/`).

In [ ]:
# Reference resources
HUMAN_TF_PATH = Path("/new-stg/home/alex/ref/gene_sets/hs_tfs.txt")
LIGAND_RECEPTOR_PATH = Path("/new-stg/home/alex/ref/cellchatdb/hs/interaction_input_CellChatDB.csv")
VALID_TARGETS_PATH = Path("/new-stg/home/alex/tmp/grn_inference_v2/reproduction/inputs/external/ref/genome/hg38/annotations/protein_coding_genes.csv")

# tradeSeq association tables (with local-FDR columns) for each cell context
EXTERNAL_ANALYSES = Path("/new-stg/home/alex/tmp/grn_inference_v2/reproduction/inputs/external/projects/chrono_seq/analyses")
CONTEXTS = {
    "K562": {
        "stimulus": "TNFa",
        "source": "TNF",
        "is_ligand": True,
        "gene_table": EXTERNAL_ANALYSES / "k562-tnfa.hg38.rep/intmd/sc-K562-1e4_to_tradeSeq/asso_df.1trajectory.lfdr.csv",
    },
    "THP1": {
        "stimulus": "TLR4",
        "source": "TLR4",
        "is_ligand": False,
        "gene_table": EXTERNAL_ANALYSES / "thp1-k562-lps.hg38.rep/intmd/sc-THP1_to_tradeSeq/asso_df.1trajectory.lfdr.csv",
    },
}

OUTPUT_DIR = Path("outputs")
TF_RANK_DIR = OUTPUT_DIR / "omnipath_tf_ranks"
TARGET_SCORE_DIR = OUTPUT_DIR / "tables"
EDGE_DIR = OUTPUT_DIR / "tables" / "chronopack_grn"


## 1. Rank TFs with OmniPath

Mass starts at each context's upstream signal (for a ligand, at its receptors from the ligand-receptor table), spreads along OmniPath edges (flipping sign across inhibitory edges) and is absorbed when it reaches a TF. `alpha` damps the mass at each step; `w_signed` / `w_unsigned` weight edges with and without a known sign. TFs are ranked by the net activating mass they absorb. Writes `omnipath_ranks.<stimulus>.csv`.

In [ ]:
tf_result = cp.tl.grn.rank_regulators(
    human_tf_path=HUMAN_TF_PATH,
    ligand_receptor_path=LIGAND_RECEPTOR_PATH,
    contexts={
        spec["stimulus"]: {"source": spec["source"], "is_ligand": spec["is_ligand"]}
        for spec in CONTEXTS.values()
    },
    output_dir=TF_RANK_DIR,
    w_signed=1.0,
    w_unsigned=0.5,
    alpha=0.85,
    max_steps=80,
    min_steps=10,
    tol_mass=1e-12,
)
tf_result.write(TF_RANK_DIR)


In [ ]:
for context, spec in CONTEXTS.items():
    ranking = tf_result.rankings[spec["stimulus"]]
    print(f"{context} ({spec['stimulus']}): {len(ranking)} TFs ranked")
    display(ranking.head(10))


## 2. Score target genes

Recalibrates each context's tradeSeq association p-values into local FDRs (fdrtool / beta-uniform mixture) over protein-coding target genes. Writes `thp1_k562_lfdr_recalibration_{scores,summary,params}.csv`.

In [ ]:
target_scores = cp.tl.grn.score_targets(
    valid_targets_path=VALID_TARGETS_PATH,
    gene_tables={context: spec["gene_table"] for context, spec in CONTEXTS.items()},
    output_dir=TARGET_SCORE_DIR,
)
target_scores.sidecar_paths


In [ ]:
print("scores", target_scores.scores.shape)
display(target_scores.summary)
display(target_scores.scores.head())


## 3. Infer TF → target edges

Combines each context's TF priorities (`priority_act`) with its target-gene local FDRs (`lfdr_bum`). Methods are named `ChronoPack_<context>`. Benchmark-specific TF-universe filtering is intentionally not applied here.

In [ ]:
grn = cp.tl.grn.infer(
    gene_scores=target_scores.scores,
    tf_scores={f"ChronoPack_{context}": tf_result.rankings[spec["stimulus"]] for context, spec in CONTEXTS.items()},
    contexts={f"ChronoPack_{context}": context for context in CONTEXTS},
    valid_targets=VALID_TARGETS_PATH,
    provenance={"workflow": "grn_inference/GRNInference.ipynb"},
)
grn.write(EDGE_DIR, prefix="chronopack_grn")


In [ ]:
print("edges", grn.edges.shape)
print("regulators", grn.edges["regulator"].nunique(), "| targets", grn.edges["target"].nunique())
display(grn.edges.groupby("method").size().rename("edges"))
display(grn.edges.head(10))
